# Model preprocessing — surgery duration

This notebook creates a leakage-conscious dataset for predicting operating-room occupancy duration. It assumes prediction occurs while scheduling the intervention, before the actual operating-room timestamps exist. It does not encode categorical variables; encoding belongs inside the model pipeline and must be fitted on the training split only.

Outputs: `resources/model_surgery_duration_dataset.parquet` and its training-only category mapping, `resources/model_surgery_duration_dataset.preprocessing.joblib`. Feature engineering is shared with inference through `duration_features.py`.

> Privacy: row-level identifiers are retained only as `split_patient_id` for split auditing in the ignored `resources/` directory. They must never be used as model features or displayed.

## Prediction-time assumptions and leakage controls

The target is `room_duration_min`, derived from actual room entry and exit. A row is eligible only when both `Heure d'entrée en salle d'opération (calimed)` and `Heure de sortie de salle d'opération (calimed)` are present and non-zero. Missing or zero-coded endpoints are excluded rather than imputed. Plausible long procedures are retained rather than automatically clipped.

The feature set deliberately excludes every raw or converted timestamp, all derived operating-room durations, stay and discharge information, GHM/GHS groupings, case ID, and patient ID. Principal CIM and CCAM codes are included under the assumption that planned diagnosis and procedure codes are available when scheduling. Set `INCLUDE_PLANNED_CLINICAL_CODES = False` if that is not true locally.

In [ ]:
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
from IPython.display import display

DATA_FILE = Path("resources/donnees_bloc_pretraitees.parquet")
OUTPUT_FILE = Path("resources/model_surgery_duration_dataset.parquet")
INCLUDE_PLANNED_CLINICAL_CODES = True
MIN_CATEGORY_COUNT = 10
VALIDATION_START = pd.Timestamp("2021-01-01")
TEST_START = pd.Timestamp("2022-01-01")
MISSING_LABEL = "__MISSING__"
RARE_LABEL = "__RARE__"

assert DATA_FILE.exists(), (
    f"Missing {DATA_FILE}. Run EDA_donees_bloc.ipynb from top to bottom first."
)

In [ ]:
df = pd.read_parquet(DATA_FILE, engine="pyarrow")
required_columns = {
    "id_patient", "date_inter", "age_years", "sexe", "interv_type",
    "anesth_type", "anesth_loco_reg", "praticien", "nom_chir",
    "cim_diag_pr", "ccam_1", "room_in_minute", "room_out_minute",
    "room_duration_min",
}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
print(f"Loaded {len(df):,} cases and {df.shape[1]} columns.")

## 1. Define and validate the target

Missing and zero room-entry or room-exit times are treated as unusable recording values. They are never imputed because those endpoints define the prediction target. Long durations remain in the data and are summarized for domain review. Any later clipping or target transformation must be chosen using the training partition only.

In [ ]:
target = pd.to_numeric(df["room_duration_min"], errors="coerce")
room_in = pd.to_numeric(df["room_in_minute"], errors="coerce")
room_out = pd.to_numeric(df["room_out_minute"], errors="coerce")

missing_room_in_mask = room_in.isna()
missing_room_out_mask = room_out.isna()
zero_room_in_mask = room_in.eq(0)
zero_room_out_mask = room_out.eq(0)
invalid_room_time_mask = (
    missing_room_in_mask
    | missing_room_out_mask
    | zero_room_in_mask
    | zero_room_out_mask
)
valid_target_mask = (
    target.notna()
    & target.gt(0)
    & target.le(24 * 60)
    & ~invalid_room_time_mask
)
eligible = df.loc[valid_target_mask].copy()
eligible["target_surgery_duration_min"] = target.loc[valid_target_mask].astype("float64")
eligible["target_surgery_duration_log1p"] = np.log1p(
    eligible["target_surgery_duration_min"]
)

assert room_in.loc[valid_target_mask].notna().all() and room_in.loc[valid_target_mask].ne(0).all()
assert room_out.loc[valid_target_mask].notna().all() and room_out.loc[valid_target_mask].ne(0).all()

quality_summary = pd.Series({
    "Initial cases": len(df),
    "Missing room-entry time": missing_room_in_mask.sum(),
    "Missing room-exit time": missing_room_out_mask.sum(),
    "Zero-coded room-entry time": zero_room_in_mask.sum(),
    "Zero-coded room-exit time": zero_room_out_mask.sum(),
    "Rows excluded for invalid room endpoints (combined)": invalid_room_time_mask.sum(),
    "Other invalid duration targets": (~valid_target_mask & ~invalid_room_time_mask).sum(),
    "Eligible cases": len(eligible),
}, name="cases").to_frame()
display(quality_summary)
display(eligible["target_surgery_duration_min"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
).to_frame("room duration (minutes)"))

## 2. Engineer scheduling-time features

Free-text labels are normalized for case, accents, and repeated whitespace. Missing clinical information remains explicit. Counts summarize the presence of secondary diagnoses and procedures without inventing codes. Actual room timing, incision timing, preoperative waiting, and length of stay are never used as predictors.

In [ ]:
from duration_features import engineer_features
categorical_features = [
    "sexe", "interv_type", "anesth_type", "anesth_loco_reg", "praticien", "nom_chir",
]
if INCLUDE_PLANNED_CLINICAL_CODES:
    categorical_features += ["cim_diag_pr", "cim_diag_family", "ccam_1", "ccam_1_family"]
numeric_features = [
    "age_years", "intervention_month_sin", "intervention_month_cos",
    "intervention_weekday_sin", "intervention_weekday_cos", "intervention_is_weekend",
    "num_secondary_diagnoses", "num_ccam_codes", "anesth_type_missing", "anesth_loco_reg_missing",
]
intervention_date = pd.to_datetime(eligible["date_inter"], errors="coerce")
# Missing dates are excluded by the temporal split, as before.
features = engineer_features(eligible.loc[intervention_date.notna()],
                             numeric_features + categorical_features).reindex(eligible.index)
# Retain year for auditing only; it is excluded from model predictors.
features["intervention_year"] = intervention_date.dt.year.astype("Int16")
print(f"Prepared {len(numeric_features)} numeric and {len(categorical_features)} categorical predictors.")


## 3. Create patient-disjoint temporal splits

Training uses interventions before 2021, validation uses new patients first represented in 2021, and testing uses new patients first represented from 2022 onward. Later records for patients already assigned to an earlier period are excluded. This conservative rule prevents the same patient from appearing in multiple splits while preserving future-data evaluation.

In [ ]:
def patient_disjoint_temporal_split(patient_ids, event_dates):
    split = pd.Series("excluded_cross_period", index=patient_ids.index, dtype="string")

    train_mask = event_dates.lt(VALIDATION_START)
    split.loc[train_mask] = "train"
    train_patients = set(patient_ids.loc[train_mask].dropna())

    validation_period = event_dates.ge(VALIDATION_START) & event_dates.lt(TEST_START)
    validation_mask = validation_period & ~patient_ids.isin(train_patients)
    split.loc[validation_mask] = "validation"
    validation_patients = set(patient_ids.loc[validation_mask].dropna())

    test_period = event_dates.ge(TEST_START)
    earlier_patients = train_patients | validation_patients
    test_mask = test_period & ~patient_ids.isin(earlier_patients)
    split.loc[test_mask] = "test"
    return split


split = patient_disjoint_temporal_split(eligible["id_patient"], intervention_date)
retained_mask = split.isin(["train", "validation", "test"])

model_data = pd.DataFrame({
    "split": split.loc[retained_mask],
    "split_patient_id": eligible.loc[retained_mask, "id_patient"],
    "split_event_date": intervention_date.loc[retained_mask],
})
model_data = pd.concat(
    [
        model_data,
        features.loc[retained_mask],
        eligible.loc[retained_mask, [
            "target_surgery_duration_min",
            "target_surgery_duration_log1p",
        ]],
    ],
    axis=1,
)

print(f"Excluded cross-period repeat-patient cases: {(~retained_mask).sum():,}")
display(model_data["split"].value_counts().rename("cases").to_frame())

## 4. Group rare categories using training data only

Categories occurring fewer than `MIN_CATEGORY_COUNT` times in training become `__RARE__`. Validation/test-only values therefore become rare rather than leaking future frequency information into preprocessing. Missing values remain a separate explicit category.

In [ ]:
from duration_features import fit_category_mapping, apply_category_mapping
train_mask = model_data["split"].eq("train")
category_mapping = fit_category_mapping(model_data.loc[train_mask], categorical_features,
                                        min_count=MIN_CATEGORY_COUNT)
rare_summary = []
for column in categorical_features:
    frequent_values = category_mapping[column]
    rare_mask = ~model_data[column].isin(frequent_values) & model_data[column].ne(MISSING_LABEL)
    rare_summary.append({
        "feature": column,
        "training_categories": model_data.loc[train_mask, column].nunique(),
        "retained_categories": len(frequent_values),
        "rows_grouped_as_rare": int(rare_mask.sum()),
    })
model_data = apply_category_mapping(model_data, category_mapping)
display(pd.DataFrame(rare_summary).set_index("feature"))


## 5. Validate and export

The `split_*` columns are metadata, not predictors. They remain in the local Parquet file so teammates can reproduce and audit the split.

In [ ]:
split_patient_sets = {
    name: set(group["split_patient_id"])
    for name, group in model_data.groupby("split")
}
assert split_patient_sets["train"].isdisjoint(split_patient_sets["validation"])
assert split_patient_sets["train"].isdisjoint(split_patient_sets["test"])
assert split_patient_sets["validation"].isdisjoint(split_patient_sets["test"])
assert model_data["target_surgery_duration_min"].gt(0).all()
assert not model_data[categorical_features].isna().any().any()
leakage_columns = {
    "room_duration_min", "entry_to_incision_min", "preop_wait_min",
    "room_in_minute", "room_out_minute", "incision_minute",
    "date_sortie", "duree_sejour_corrigee", "length_of_stay_days",
    "id_patient", "no_cas",
}
assert not leakage_columns.intersection(features.columns)

split_summary = model_data.groupby("split").agg(
    cases=("target_surgery_duration_min", "size"),
    patients=("split_patient_id", "nunique"),
    median_room_min=("target_surgery_duration_min", "median"),
    p90_room_min=("target_surgery_duration_min", lambda values: values.quantile(0.9)),
    p99_room_min=("target_surgery_duration_min", lambda values: values.quantile(0.99)),
)
display(split_summary.round(1))

model_data = model_data.reset_index(drop=True)
model_data.to_parquet(OUTPUT_FILE, index=False, engine="pyarrow")
print(f"Saved {len(model_data):,} rows and {model_data.shape[1]} columns to {OUTPUT_FILE}")
MAPPING_FILE = OUTPUT_FILE.with_suffix(".preprocessing.joblib")
joblib.dump(category_mapping, MAPPING_FILE)
print(f"Saved training-only category mapping to {MAPPING_FILE}")


## 6. Model-ingestion example

Use `numeric_features + categorical_features` as the feature list. Keep encoding, imputation, scaling, and any target transformation inside a fitted model pipeline. Never include columns beginning with `split_`, and train against only one duration target representation.

In [ ]:
FEATURE_COLUMNS = numeric_features + categorical_features
train = model_data[model_data["split"].eq("train")]
validation = model_data[model_data["split"].eq("validation")]
test = model_data[model_data["split"].eq("test")]

X_train = train[FEATURE_COLUMNS]
y_train = train["target_surgery_duration_min"]
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)